# xQTL Model Training Tutorial

This notebook implements the sc-EMS (single-cell Expression Modifier Scores) methodology from our previous manuscript. The original research demonstrated that machine learning can predict which genetic variants affect gene expression in specific brain cell types, particularly for Alzheimer's disease research.



### Current Goal
We want to take the broad feature categories identified in our manuscript (shown in Figure 1, Table C on page 10) and create a **clean, reproducible pipeline** that researchers can apply to **any genetic variant**. 

### Motivation
The original research showed that:
- Traditional methods only explain 5% of Alzheimer's disease heritability
- sc-EMS predictions explain **26% of disease heritability** 
- Cell-type specific effects (especially in microglia) were previously invisible in bulk tissue studies

### Aim
Transform the complex research findings into a standardized tool that:
1. Takes any genetic variant as input
2. Uses the proven feature categories from Table C
3. Outputs cell-type specific effect predictions
4. Is reproducible across different research groups


## Methods Overview

### High-Level Approach
- Our approach uses a streamlined pipeline with cell-type specific modeling and feature weighting based on proven methodology from our manuscript. The pipeline focuses on four CatBoost models with different regularization strategies to ensure robust predictions across diverse genomic contexts.
  
- ### CatBoost Algorithm
**CatBoost** (Categorical Boosting) is a gradient boosting machine learning algorithm specifically designed for datasets with categorical features and mixed data types. It builds an ensemble of decision trees sequentially, where each new tree corrects errors made by previous trees.


## Input Data: Feature Categories from Manuscript Table C

Based on our previous research (Figure 1, Table C), we use these broad feature categories. Each category represents a branch of features that can be applied to many different contexts and datasets.


In [ ]:
# Load gene constraint data
import pandas as pd
gene_data = pd.read_excel('data/41588_2024_1820_MOESM4_ESM.xlsx')
print(gene_data.head(3))

**Sample output showing GeneBayes constraint scores:**
```
ensg            hgnc        chrom  obs_lof  exp_lof   post_mean    post_lower_95  post_upper_95
ENSG00000198488 HGNC:24141  chr11    12      8.9777   6.46629E-05  7.16256E-06   0.00017805
ENSG00000164363 HGNC:26441  chr5     31      28.55    0.00016062   2.59918E-05   0.00044175
ENSG00000159337 HGNC:30038  chr15    28      41.84    0.00016978   0.000018674   0.00053317
```

### 1. **Distance Features**
 **Brief Description**: This broad category captures spatial relationships between genetic variants and genes in the genome.
 - Can be expanded to include distances to any genomic feature of interest (splice sites, CTCF binding sites, topological domain boundaries), making it adaptable to different research questions and datasets
### 2. **Non Cell-Type Specific Features**
**Brief Description**: This category encompasses fundamental genomic annotations that apply universally across all cell types and tissues.
- Can incorporate any genome-wide annotation dataset, including new conservation metrics, population genetics data, or functional predictions that do not depend on specific cellular contexts
### 3. **Cell-Type Specific Features**
**Brief Description**: This category captures regulatory annotations that vary between different cell types, representing the cellular context-dependent nature of gene regulation.
- Can be extended to any cell type or tissue of interest, incorporating new single-cell datasets, tissue-specific regulatory maps, or developmental stage-specific annotations.
### 4. **DL Enformer Features**
**Brief Description**: This category represents deep learning-based predictions of variant effects using large-scale sequence-to-function models.
- Can incorporate predictions from any deep learning model trained on genomic sequences, including newer architectures, species-specific models, or task-specific predictors.
### 5. **DL BPNet Features**
**Brief Description**: This category includes specialized deep learning predictions focused on transcription factor binding and chromatin accessibility changes.
- Can expand to include predictions from any protein-DNA interaction model, new chromatin accessibility predictors, or binding specificity models for different protein families.

### Output Data Generated
| Output Type | Description | Research Use |
|-------------|--------|-------------|--------------|
| **Trained models** | Four CatBoost classifiers | Variant effect prediction |
| **Performance metrics** | AP/AUC scores | Model comparison |
| **Feature importance** | Genomic feature rankings | Biological interpretation |
| **Predictions** | Test chromosome scores | Validation analysis |

## Key Parameters

### Critical Model Hyperparameters
| Parameter | Models 1,5 | Models 3,7 | Biological Rationale |
|-----------|------------|------------|---------------------|
| **`depth`** | 6 | 5 | Tree complexity vs. overfitting trade-off |
| **`l2_leaf_reg`** | 3.0 | 5.0 | Regularization strength for genomic noise |
| **`learning_rate`** | 0.03 | 0.03 | Conservative learning for stability |
| **`iterations`** | 1000 | 1000 | Sufficient for convergence |

### Feature Weighting Strategy
- **Standard features:** Weight = 1.0 (baseline genomic annotations)
- **High-priority features:** Weight = 10.0 (chrombpnet_positive, tf_positive, diff)
- **Rationale:** Emphasize experimentally validated regulatory signals

### Cross-Validation Configuration
- **Training:** 21 chromosomes (chr1, chr3-22)
- **Testing:** 1 chromosome (chr2)
- - **Why split this way?** Variants on the same chromosome are related, so testing on a separate chromosome gives us realistic performance estimates.

# Detailed Workflow: Step-by-Step Guide

*This section explains how to use our xQTL training pipeline*

## What We're Actually Doing

We're training 4 machine learning models to predict which genetic variants will affect gene expression. 

## Step 1: Running the Pipeline

The main command to start training:

```bash
cd ~/MLxQTL/pipeline/5_model_training/
pixi run python model_training_simplified.py Mic_mega_eQTL 2 \
  --gene_lof_file ../../data/41588_2024_1820_MOESM4_ESM.xlsx \
  --yaml_path data_params.yaml

In [ ]:
```python
# Quick check - do we have the required files?
import os

required_files = [
    'model_training_simplified.py',
    'data_params.yaml', 
    '../../data/41588_2024_1820_MOESM4_ESM.xlsx'
]

print("🔍 File Check:")
for file in required_files:
    status = "✅ Found" if os.path.exists(file) else "❌ Missing"
    print(f"  {file}: {status}")

print(f"\n📋 Training Process:")
print(f"  1. Load genomic data from 21 chromosomes")
print(f"  2. Create 4 CatBoost models with different strategies") 
print(f"  3. Train models to recognize functional variants")
print(f"  4. Test performance on chromosome 2")
print(f"  5. Save models and results")

## Step 2: Understanding the 4 Model Strategies

Our pipeline creates 4 CatBoost models, each with a different approach:

| Model | Strategy | Purpose |
|-------|----------|---------|
| **Model 1** | Standard parameters | Baseline performance |
| **Model 3** | Conservative | Reduce overfitting |
| **Model 5** | Feature weighting | Biology-informed predictions |
| **Model 7** | Conservative + weighted | Robust + biology-focused |

**Key Features That Get 10x Weighting:**
- `chrombpnet_positive` - Chromatin accessibility signals
- `tf_positive` - Transcription factor binding sites  
- `diff` - Differential expression scores

**Why 4 models?** Each has different strengths. Some are conservative (fewer false positives), others are sensitive (catch more true effects). Having multiple models gives us confidence in predictions.

## Step 3: What You Get as Output

After training completes, you'll find these files in your `model_results/` directory:

### Trained Models (4 files)
- `model_standard_subset_chr_chr2_NPR_10.joblib` - Standard model
- `model_standard_subset_conservative_chr_chr2_NPR_10.joblib` - Conservative model  
- `model_standard_subset_weighted_chr_chr2_NPR_10.joblib` - Weighted model
- `model_standard_subset_conservative_weighted_chr_chr2_NPR_10.joblib` - Conservative+weighted

### Analysis Results
- `summary_dict_catboost_4models_chr_chr2_NPR_10.pkl` - Performance metrics / How well did each model do? 
- `features_importance_4models_chr_chr2_NPR_10.csv` - What did the models pay attention to? 
- `predictions_4models_chr2.tsv` - Actual predictions on test chromosome

In [ ]:
```python
# Let's show what you might actually see when running the pipeline
print("📊 Real Performance Results from Training:")
print("=" * 60)

# These are the actual results you got
actual_performance = {
    'Model 1 (Standard)': {'AP': 0.2194, 'AUC': 0.5009},
    'Model 3 (Conservative)': {'AP': 0.2194, 'AUC': 0.5009}, 
    'Model 5 (Weighted)': {'AP': 0.2194, 'AUC': 0.5009},
    'Model 7 (Conservative+Weighted)': {'AP': 0.2194, 'AUC': 0.5009}
}

print("Actual Results:")
for model, metrics in actual_performance.items():
    print(f"  {model}:")
    print(f"    Average Precision: {metrics['AP']:.4f}")
    print(f"    AUC Score: {metrics['AUC']:.4f}")

print(f"\n🤔 What These Scores Mean:")
print(f"  • AUC ≈ 0.50 = Random performance (like flipping a coin)")
print(f"  • AP ≈ 0.22 = Below baseline (concerning)")
print(f"  • Identical scores = Models learned the same pattern")

print(f"\n🔍 Feature Importance Pattern:")
print(f"  • gene_lof: 100% importance")
print(f"  • All other features: 0% importance")
print(f"  • This suggests the model is only using gene LOF scores!")

print(f"\n⚠️  This indicates potential data issues:")
print(f"  1. Training/test data imbalance")
print(f"  2. Feature engineering problems")
print(f"  3. Data quality issues")
print(f"  4. Need to check data preprocessing steps")

### What Good vs. Concerning Results Look Like

**🎯 Good Performance Indicators:**
- AUC scores > 0.75 (ideally > 0.85)
- Average Precision > 0.4 (ideally > 0.6)
- Different models showing slightly different scores
- Feature importance spread across multiple feature types

**⚠️ Concerning Signs (Like What We See Here):**
- AUC ≈ 0.50 (random performance)
- All models identical (suggests data issues)
- Only one feature (gene_lof) being used
- Other genomic features ignored


## Step 4: Using Your Trained Models

Once training is complete, load the trained models for predictions. Please refer to **EMS Predictions** (https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_prediction.html) for detailed prediction workflows and variant scoring.
